In [0]:
%pip install "databricks-labs-dqx>=0.9.0"

In [0]:
import pkg_resources
import sys
display({
    "databricks-labs-dqx version": pkg_resources.get_distribution("databricks-labs-dqx").version,
    "python version": sys.version
})

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from databricks.sdk import WorkspaceClient

from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule
from databricks.labs.dqx import check_funcs

In [0]:
# import databricks.labs.dqx
# print(help(databricks.labs.dqx))

In [0]:
df = table('dqx_sandbox.dqx_demo.customer')

In [0]:
rules = [
    DQRowRule(
        name="customer_id_not_null",
        column="customer_id",
        check_func=check_funcs.is_not_null,
        criticality="error" 
    )
]

# 4. Initialize Engine and Run
ws = WorkspaceClient()
engine = DQEngine(ws)

# This returns: (Passed Rows, Failed/Quarantined Rows)
valid_and_quarantine_df = engine.apply_checks(df, rules)

# 5. Review Results
print("--- Quarantine Report ---")
display(valid_and_quarantine_df)

In [0]:
%sql
desc detail dqx_sandbox.dqx_demo.customer 

In [0]:
%sql
select * from dqx_sandbox.dqx_bronze.customer

In [0]:
import yaml

In [0]:
checks = yaml.safe_load(
    """
    - criticality: error
      check:
        function: foreign_key
        arguments:
          columns: 
          - col1
          ref_columns: 
          - ref_col1
          # either provide reference DataFrame name
          ref_df_name: ref_df_key
          # or provide name of the reference table
          #ref_table: catalog1.schema1.ref_table
  
    - criticality: warn
      name: foreign_key_check_on_composite_key
      check:
        function: foreign_key
        arguments:
          columns: 
          - col1
          - col2
          ref_columns:
          - ref_col1
          - ref_col2
          ref_df_name: ref_df_key
    """)


In [0]:
checks

In [0]:
%sql
SELECT 
    customer_email,
    CASE 
        WHEN customer_email REGEXP '^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$' THEN 'Pass'
        ELSE 'Fail'
    END as dq_status
FROM dqx_sandbox.dqx_bronze.customer;